<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/Numba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sum(a*b) on a CPU

In [1]:
import numpy as np
from numba import njit, prange

# 1. @njit: Compiles to machine code (No-Python mode)
# 2. parallel=True: Enables automatic multi-threading
@njit(parallel=True)
def numba_dot(a, b):
    n = a.shape[0]
    total = 0.0

    # prange is the Numba equivalent of OpenMP's '#pragma omp parallel for'
    # It tells the compiler this loop is safe to run in parallel
    for i in prange(n):
        total += a[i] * b[i]

    return total

# --- Execution ---
n = 1_000_000
a = np.ones(n, dtype=np.float64)
b = np.full(n, 2.0, dtype=np.float64)

# First call: Compiles the function (slight delay)
# Second call: Runs at the speed of C++/Fortran
result = numba_dot(a, b)

print(f"Numba Result: {result}")

Numba Result: 2000000.0


# Sum(a*b) on a GPU

In [1]:
from numba import cuda
import numpy as np
import math

# 1. Define the CUDA Kernel
@cuda.jit
def gpu_multiply(a, b, result):
    # Calculate the unique thread index in the 1D grid
    idx = cuda.grid(1)

    # Check boundary to avoid memory errors
    if idx < a.size:
        result[idx] = a[idx] * b[idx]

# --- Execution ---
n = 1_000_000
a_cpu = np.ones(n, dtype=np.float32)
b_cpu = np.full(n, 2.0, dtype=np.float32)
res_cpu = np.zeros(n, dtype=np.float32)

# 2. Transfer data to the GPU (Host to Device)
a_gpu = cuda.to_device(a_cpu)
b_gpu = cuda.to_device(b_cpu)
res_gpu = cuda.device_array_like(res_cpu)

# 3. Configure the Grid
threads_per_block = 256
blocks_per_grid = math.ceil(n / threads_per_block)

# 4. Launch the Kernel
gpu_multiply[blocks_per_grid, threads_per_block](a_gpu, b_gpu, res_gpu)

# 5. Copy result back to CPU (Device to Host)
res_cpu = res_gpu.copy_to_host()
final_sum = np.sum(res_cpu)

print(f"GPU Multiply-Sum Result: {final_sum}")

GPU Multiply-Sum Result: 2000000.0


In [2]:
from numba import cuda
import numpy as np

# 1. Define the Block Size (Must be a power of 2 for easy reduction)
TPB = 256

@cuda.jit
def dot_product_shared(a, b, partial_sums):
    # 's_mem' is the fast workbench shared by all threads in this block
    s_mem = cuda.shared.array(shape=(TPB,), dtype=cuda.float32)

    tx = cuda.threadIdx.x
    idx = cuda.grid(1)

    # Load data into Shared Memory
    if idx < a.size:
        s_mem[tx] = a[idx] * b[idx]
    else:
        s_mem[tx] = 0.0

    # Wait for all threads in the block to finish loading
    cuda.syncthreads()

    # Parallel Reduction (Summing inside Shared Memory)
    # We fold the array in half repeatedly: 128->64->32...
    stride = TPB // 2
    while stride > 0:
        if tx < stride:
            s_mem[tx] += s_mem[tx + stride]
        cuda.syncthreads() # Sync after every fold!
        stride //= 2

    # Thread 0 of each block writes the FINAL block-sum to global memory
    if tx == 0:
        partial_sums[cuda.blockIdx.x] = s_mem[0]

# --- Execution ---
n = 1_024_000 # Multiple of 256
a_gpu = cuda.to_device(np.ones(n, dtype=np.float32))
b_gpu = cuda.to_device(np.full(n, 2.0, dtype=np.float32))

blocks = n // TPB
partial_sums_gpu = cuda.device_array(blocks, dtype=np.float32)

# Launch
dot_product_shared[blocks, TPB](a_gpu, b_gpu, partial_sums_gpu)

# Final step: Sum the few hundred 'partial sums' on the CPU
final_result = np.sum(partial_sums_gpu.copy_to_host())
print(f"Result with Shared Memory: {final_result}")

Result with Shared Memory: 2048000.0


# N particle on a GPU


In [6]:
from numba import cuda
import numpy as np
import math

# 1. Constants
TPB = 256  # Threads per Block (Tile Size)
G = 1.0
SOFTENING = 0.1

@cuda.jit
def nbody_tiled_kernel(pos, masses, out_forces):
    # Shared memory 'tile' for particle positions and masses
    # Every thread in the block will help fill this workbench
    sh_pos = cuda.shared.array(shape=(TPB, 3), dtype=cuda.float32)
    sh_mass = cuda.shared.array(shape=(TPB,), dtype=cuda.float32)

    tx = cuda.threadIdx.x
    idx = cuda.grid(1)
    N = pos.shape[0]

    # Each thread tracks its own particle's position and accumulated force
    local_pos = cuda.local.array(3, dtype=cuda.float32)
    if idx < N:
        for d in range(3):
            local_pos[d] = pos[idx, d]

    accel = cuda.local.array(3, dtype=cuda.float32)
    for d in range(3):
        accel[d] = 0.0

    # Loop over all 'tiles' of source particles
    num_tiles = (N + TPB - 1) // TPB
    for t in range(num_tiles):
        # Collaborative Load: Each thread loads one particle into shared memory
        tile_idx = t * TPB + tx
        if tile_idx < N:
            for d in range(3):
                sh_pos[tx, d] = pos[tile_idx, d]
            sh_mass[tx] = masses[tile_idx]
        else:
            for d in range(3):
                sh_pos[tx, d] = 0.0
            sh_mass[tx] = 0.0

        # Synchronization: Ensure the entire tile is loaded before calculation
        cuda.syncthreads()

        # Compute interactions with the current tile in shared memory
        for j in range(TPB):
            dx = sh_pos[j, 0] - local_pos[0]
            dy = sh_pos[j, 1] - local_pos[1]
            dz = sh_pos[j, 2] - local_pos[2]

            dist_sq = dx*dx + dy*dy + dz*dz + SOFTENING**2
            inv_dist_cube = 1.0 / math.sqrt(dist_sq**3)

            s = sh_mass[j] * inv_dist_cube
            accel[0] += s * dx
            accel[1] += s * dy
            accel[2] += s * dz

        # Synchronization: Ensure calculation is done before loading next tile
        cuda.syncthreads()

    # Write final force back to global memory
    if idx < N:
        for d in range(3):
            out_forces[idx, d] = G * accel[d]

# --- Setup ---
N = 1024*64
pos = np.random.randn(N, 3).astype(np.float32)
masses = np.random.rand(N).astype(np.float32)
forces = np.zeros_like(pos)

# Launch
blocks = (N + TPB - 1) // TPB
nbody_tiled_kernel[blocks, TPB](cuda.to_device(pos), cuda.to_device(masses), cuda.to_device(forces))

# Derivative in 1D:

Consider 1d array with 2**16 elements. Write a warp function to create 1st derivative using central difference. FOr the left end points use forward and for the right end point use backward diff.

In [1]:
from numba import cuda
import numpy as np

# Block size
TPB = 256

@cuda.jit
def central_diff_kernel(f, dx, df):
    # Shared memory size = Block size + 2 (for the Halo cells)
    # We use a 1-element offset for indexing
    s_f = cuda.shared.array(shape=(TPB + 2,), dtype=cuda.float32)

    tx = cuda.threadIdx.x
    idx = cuda.grid(1)
    n = f.size

    # 1. Collaborative Load into Shared Memory
    # Shared memory index 'tx + 1' corresponds to global index 'idx'
    if idx < n:
        s_f[tx + 1] = f[idx]

        # Load Left Halo
        if tx == 0:
            s_f[0] = f[idx - 1] if idx > 0 else 0.0

        # Load Right Halo
        if tx == TPB - 1 or idx == n - 1:
            s_f[tx + 2] = f[idx + 1] if idx < n - 1 else 0.0

    cuda.syncthreads()

    # 2. Compute Derivative
    if idx < n:
        # LEFT BOUNDARY: Forward Difference
        if idx == 0:
            df[idx] = (s_f[tx + 2] - s_f[tx + 1]) / dx

        # RIGHT BOUNDARY: Backward Difference
        elif idx == n - 1:
            df[idx] = (s_f[tx + 1] - s_f[tx]) / dx

        # INTERIOR: Central Difference
        else:
            # (f[i+1] - f[i-1]) / 2dx
            df[idx] = (s_f[tx + 2] - s_f[tx]) / (2.0 * dx)

# --- Execution ---
N = 2**16
dx = 1.0 / N
x = np.linspace(0, 1, N, dtype=np.float32)
f_cpu = np.sin(2 * np.pi * x) # Example function
df_cpu = np.zeros_like(f_cpu)

# Transfer to GPU
f_gpu = cuda.to_device(f_cpu)
df_gpu = cuda.device_array_like(f_cpu)

blocks = (N + TPB - 1) // TPB
central_diff_kernel[blocks, TPB](f_gpu, dx, df_gpu)

# Verify Result
df_final = df_gpu.copy_to_host()

TypingError: Failed in cuda mode pipeline (step: nopython frontend)
No implementation of function Function(<function shared.array at 0x7b671d6842c0>) found for signature:
 
 >>> array(shape=UniTuple(int64 x 1), dtype=class(float32))
 
There are 2 candidate implementations:
  - Of which 2 did not match due to:
  Overload of function 'array': File: numba/cuda/cudadecl.py: Line 28.
    With argument(s): '(shape=UniTuple(int64 x 1), dtype=class(float32))':
   No match.

During: resolving callee type: Function(<function shared.array at 0x7b671d6842c0>)
During: typing of call at /tmp/ipython-input-11524/2687678777.py (11)


File "../tmp/ipython-input-11524/2687678777.py", line 11:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference